# Dagbani ASR Phase 1B — GPU Baseline Only

This auxiliary Kaggle notebook evaluates `openai/whisper-small` and the
public WAXAL Dagbani Whisper-small checkpoint without repeating the CPU data
audit. It reads the release-ready private Phase 1 manifest, freezes the
WAXAL cleaning filter (duration >= 1.5 seconds and speech rate <= 4
words/second), batches GPU inference, and publishes predictions and metrics
back to `phase1/baselines/` in the private dataset repository.

Start with the 256-row pilot. The public checkpoint's 34.0% WER / 11.9% CER
target applies to the full filtered evaluation set, not necessarily the pilot.
The public model card currently prints `speech rate >= 4 WPS`; applied to the
sealed manifest that retains only two rows. This notebook therefore records
the empirically corrected `<= 4 WPS` cleaning rule as a separate v2 protocol.
Do not call it an exact public-score reproduction unless the official cleaned
evaluation manifest or filtering script confirms the same rows.

## 1. Install only missing runtime packages

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "datasets": "datasets[audio]>=3.2,<5",
    "transformers": "transformers>=4.48,<6",
    "huggingface_hub": "huggingface_hub>=0.27,<2",
    "jiwer": "jiwer>=3.0,<5",
    "soundfile": "soundfile>=0.12,<1",
    "librosa": "librosa>=0.10,<1",
    "pyarrow": "pyarrow>=17,<25",
}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Baseline environment ready. PyTorch is intentionally not replaced.")

## 2. Configuration — the pilot is opt-in

In [ ]:
from __future__ import annotations

import gc
import hashlib
import io
import json
import os
import string
import time
import unicodedata
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterator

import numpy as np
import pandas as pd
import soundfile as sf


def kaggle_secret(name: str) -> str | None:
    value = os.getenv(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


@dataclass
class BaselineConfig:
    manifest_repo_id: str = "ats-tech/dagbani-asr-phase1"
    hf_token: str | None = field(default_factory=lambda: kaggle_secret("HF_TOKEN"))
    model_ids: tuple[str, ...] = (
        "waxal-benchmarking/whisper-small-waxal-dag",
        "openai/whisper-small",
    )
    run_baselines: bool = False
    max_samples: int | None = 256  # Set None only after the pilot succeeds.
    batch_size: int = 8
    upload_every: int = 100
    session_budget_minutes: float = 150.0
    work_dir: str = "/kaggle/working/dagbani_asr_baselines"
    min_duration_s: float = 1.5
    max_words_per_s: float = 4.0
    protocol_id: str = "waxal-clean-v2-max4wps"

    def validate(self) -> None:
        if not self.hf_token:
            raise ValueError("Attach the Kaggle HF_TOKEN secret")
        if not self.manifest_repo_id:
            raise ValueError("manifest_repo_id is required")
        if self.batch_size < 1:
            raise ValueError("batch_size must be positive")
        if self.max_words_per_s <= 0:
            raise ValueError("max_words_per_s must be positive")


cfg = BaselineConfig()
cfg.validate()
WORK_DIR = Path(cfg.work_dir)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg) | {"hf_token": "<set>"}, indent=2))

## 3. Load and freeze the release-ready evaluation manifest

In [ ]:
from huggingface_hub import HfApi, hf_hub_download


def artifact(filename: str) -> Path:
    return Path(hf_hub_download(
        repo_id=cfg.manifest_repo_id,
        filename=f"phase1/{filename}",
        repo_type="dataset",
        token=cfg.hf_token,
        cache_dir=str(WORK_DIR / "hf_cache"),
    ))


readiness = json.loads(artifact("readiness.json").read_text(encoding="utf-8"))
if not readiness.get("release_ready"):
    raise RuntimeError(f"Phase 1 is not release-ready: {readiness}")

manifest = pd.read_parquet(artifact("accepted_manifest.parquet"))
waxal_test = manifest[
    manifest["source"].eq("waxal_dag_asr") & manifest["split"].eq("test")
].copy()
if "words_per_s" not in waxal_test:
    waxal_test["words_per_s"] = (
        waxal_test["transcript_eval"].fillna("").astype(str).str.split().str.len()
        / pd.to_numeric(waxal_test["duration_s"], errors="coerce")
    )
frozen_eval = waxal_test[
    pd.to_numeric(waxal_test["duration_s"], errors="coerce").ge(cfg.min_duration_s)
    & pd.to_numeric(waxal_test["words_per_s"], errors="coerce").le(cfg.max_words_per_s)
].sort_values("sample_id").reset_index(drop=True)
if cfg.max_samples is not None:
    frozen_eval = frozen_eval.head(cfg.max_samples).copy()
if frozen_eval.empty:
    raise RuntimeError("No WAXAL test rows passed the WAXAL cleaning filter")

repo_info = HfApi(token=cfg.hf_token).repo_info(cfg.manifest_repo_id, repo_type="dataset")
phase1_repo_head = str(repo_info.sha)
manifest_revision = hashlib.sha256(
    pd.util.hash_pandas_object(manifest, index=True).values.tobytes()
).hexdigest()
eval_manifest_hash = hashlib.sha256(
    pd.util.hash_pandas_object(
        frozen_eval[["sample_id", "transcript_raw", "audio_hash"]], index=False
    ).values.tobytes()
).hexdigest()
frozen_eval.to_parquet(WORK_DIR / "frozen_eval_manifest.parquet", index=False)
print({
    "phase1_repo_head": phase1_repo_head,
    "phase1_manifest_hash": manifest_revision,
    "sealed_test_rows": len(waxal_test),
    "filtered_eval_rows": len(frozen_eval),
    "speech_rate_rule": f"words_per_s <= {cfg.max_words_per_s}",
    "eval_manifest_hash": eval_manifest_hash,
})

## 4. Conservative metrics and audio decoding

In [ ]:
import jiwer

PUNCT_TRANSLATION = str.maketrans({char: " " for char in string.punctuation + "“”‘’…–—"})


def strict_text(value: Any) -> str:
    return " ".join(unicodedata.normalize("NFC", str(value or "")).split())


def normalize_eval(value: Any) -> str:
    return " ".join(strict_text(value).lower().translate(PUNCT_TRANSLATION).split())


def asr_metrics(references: list[str], hypotheses: list[str]) -> dict[str, float]:
    if len(references) != len(hypotheses) or not references:
        raise ValueError("References and hypotheses must be non-empty and aligned")
    strict_refs = [strict_text(value) for value in references]
    strict_hyps = [strict_text(value) for value in hypotheses]
    norm_refs = [normalize_eval(value) for value in references]
    norm_hyps = [normalize_eval(value) for value in hypotheses]
    return {
        "strict_wer": float(jiwer.wer(strict_refs, strict_hyps)),
        "strict_cer": float(jiwer.cer(strict_refs, strict_hyps)),
        "normalized_wer": float(jiwer.wer(norm_refs, norm_hyps)),
        "normalized_cer": float(jiwer.cer(norm_refs, norm_hyps)),
        "utterances": len(references),
    }


def audio_payload(value: Any) -> tuple[np.ndarray, int]:
    if hasattr(value, "get_all_samples"):
        samples = value.get_all_samples()
        array = samples.data.detach().cpu().numpy()
        sample_rate = int(samples.sample_rate)
    elif isinstance(value, dict) and value.get("array") is not None:
        array = np.asarray(value["array"], dtype=np.float32)
        sample_rate = int(value.get("sampling_rate") or 16_000)
    elif isinstance(value, dict) and value.get("bytes"):
        array, sample_rate = sf.read(io.BytesIO(value["bytes"]), dtype="float32", always_2d=False)
    else:
        raise ValueError("Unsupported WAXAL audio payload")
    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 2:
        array = array.mean(axis=0 if array.shape[0] <= 8 else 1)
    return array.reshape(-1), sample_rate


def canonical_audio(array: np.ndarray, sample_rate: int) -> np.ndarray:
    if sample_rate != 16_000:
        import librosa
        array = librosa.resample(array, orig_sr=sample_rate, target_sr=16_000)
    return np.clip(array, -1.0, 1.0).astype(np.float32)


assert asr_metrics(["N nyɛla bɛ."], ["n nyɛla bɛ"])["normalized_wer"] == 0.0
print("Metric and glyph self-test passed.")

## 5. Batched, resumable GPU inference

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

api = HfApi(token=cfg.hf_token)
eval_by_id = frozen_eval.set_index("sample_id")


def slug(model_id: str) -> str:
    return model_id.replace("/", "__")


def remote_predictions_path(model_id: str) -> str:
    mode = "full" if cfg.max_samples is None else f"pilot-{cfg.max_samples}"
    return f"phase1/baselines/{cfg.protocol_id}/{mode}/{slug(model_id)}-predictions.parquet"


def load_existing_predictions(model_id: str) -> pd.DataFrame:
    try:
        path = hf_hub_download(
            repo_id=cfg.manifest_repo_id,
            filename=remote_predictions_path(model_id),
            repo_type="dataset",
            token=cfg.hf_token,
            cache_dir=str(WORK_DIR / "resume_cache"),
            force_download=True,
        )
    except Exception:
        return pd.DataFrame(columns=[
            "sample_id", "reference", "hypothesis", "model_id",
            "manifest_revision", "eval_manifest_hash",
        ])
    table = pd.read_parquet(path)
    if not table.empty and set(table["eval_manifest_hash"].astype(str)) != {eval_manifest_hash}:
        raise RuntimeError("Remote predictions belong to a different frozen evaluation manifest")
    return table


def upload_predictions(model_id: str, table: pd.DataFrame) -> Path:
    path = WORK_DIR / f"{slug(model_id)}-predictions.parquet"
    table.to_parquet(path, index=False)
    api.upload_file(
        repo_id=cfg.manifest_repo_id,
        repo_type="dataset",
        path_or_fileobj=str(path),
        path_in_repo=remote_predictions_path(model_id),
        commit_message=f"Checkpoint Dagbani ASR baseline: {model_id} ({len(table)}/{len(frozen_eval)})",
    )
    return path


def iter_pending_audio(pending_ids: set[str]) -> Iterator[dict[str, Any]]:
    found: set[str] = set()
    stream = load_dataset(
        "google/WaxalNLP", "dag_asr", split="test", streaming=True, token=cfg.hf_token
    )
    for record in stream:
        sample_id = str(record.get("id", ""))
        if sample_id not in pending_ids:
            continue
        array, sample_rate = audio_payload(record["audio"])
        found.add(sample_id)
        yield {
            "sample_id": sample_id,
            "reference": strict_text(eval_by_id.at[sample_id, "transcript_raw"]),
            "audio": canonical_audio(array, sample_rate),
        }
    missing = pending_ids - found
    if missing:
        raise RuntimeError(f"WAXAL stream did not yield {len(missing)} frozen IDs: {sorted(missing)[:10]}")


def require_compatible_gpu() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU before running baselines")
    major, minor = torch.cuda.get_device_capability(0)
    device_arch = f"sm_{major}{minor}"
    supported_arches = set(torch.cuda.get_arch_list())
    gpu_info = {
        "gpu": torch.cuda.get_device_name(0),
        "device_arch": device_arch,
        "torch_supported_arches": sorted(supported_arches),
        "memory_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
    }
    print("GPU preflight:", gpu_info)
    if supported_arches and device_arch not in supported_arches and f"compute_{major}{minor}" not in supported_arches:
        raise RuntimeError(
            f"Kaggle's current PyTorch build does not support {device_arch}. "
            "Select T4 x2 instead of P100, restart the session, and rerun from the top."
        )


def build_model(model_id: str):
    require_compatible_gpu()
    processor = AutoProcessor.from_pretrained(model_id, token=cfg.hf_token)
    load_kwargs = {"token": cfg.hf_token, "low_cpu_mem_usage": True}
    try:
        model = AutoModelForSpeechSeq2Seq.from_pretrained(
            model_id, dtype=torch.float16, **load_kwargs
        )
    except TypeError:
        model = AutoModelForSpeechSeq2Seq.from_pretrained(
            model_id, torch_dtype=torch.float16, **load_kwargs
        )
    model.to("cuda").eval()
    model.generation_config.forced_decoder_ids = None
    return processor, model


def transcribe_batch(processor, model, batch: list[dict[str, Any]]) -> list[str]:
    inputs = processor.feature_extractor(
        [item["audio"] for item in batch],
        sampling_rate=16_000,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=30 * 16_000,
        return_attention_mask=True,
    )
    generate_kwargs = {
        "input_features": inputs.input_features.to("cuda", dtype=torch.float16),
        "task": "transcribe",
    }
    if "attention_mask" in inputs:
        generate_kwargs["attention_mask"] = inputs.attention_mask.to("cuda")
    with torch.inference_mode():
        generated_ids = model.generate(**generate_kwargs)
    return [strict_text(text) for text in processor.batch_decode(generated_ids, skip_special_tokens=True)]


def evaluate_model(model_id: str) -> dict[str, Any]:
    predictions = load_existing_predictions(model_id)
    predictions = predictions.drop_duplicates("sample_id", keep="last")
    expected_ids = set(frozen_eval["sample_id"].astype(str))
    predictions = predictions[predictions["sample_id"].astype(str).isin(expected_ids)]
    pending_ids = expected_ids - set(predictions["sample_id"].astype(str))
    started = time.time()
    processed_this_run = 0
    stopped_for_budget = False

    if pending_ids:
        processor, model = build_model(model_id)
        batch: list[dict[str, Any]] = []
        for row in iter_pending_audio(pending_ids):
            batch.append(row)
            if len(batch) < cfg.batch_size:
                continue
            hypotheses = transcribe_batch(processor, model, batch)
            new_rows = [{
                "sample_id": item["sample_id"],
                "reference": item["reference"],
                "hypothesis": hypothesis,
                "model_id": model_id,
                "manifest_revision": manifest_revision,
                "eval_manifest_hash": eval_manifest_hash,
            } for item, hypothesis in zip(batch, hypotheses)]
            predictions = pd.concat([predictions, pd.DataFrame(new_rows)], ignore_index=True)
            processed_this_run += len(new_rows)
            batch = []
            completed = predictions["sample_id"].nunique()
            elapsed = max(time.time() - started, 1e-6)
            rate = processed_this_run / elapsed
            eta_min = (len(expected_ids) - completed) / max(rate, 1e-9) / 60
            if completed % 25 < cfg.batch_size:
                print(f"{model_id}: {completed}/{len(expected_ids)}; {rate:.2f} rows/s; ETA {eta_min:.1f}m")
            if completed % cfg.upload_every < cfg.batch_size:
                upload_predictions(model_id, predictions)
            if elapsed / 60 >= cfg.session_budget_minutes:
                stopped_for_budget = True
                break

        if batch and not stopped_for_budget:
            hypotheses = transcribe_batch(processor, model, batch)
            predictions = pd.concat([predictions, pd.DataFrame([{
                "sample_id": item["sample_id"], "reference": item["reference"],
                "hypothesis": hypothesis, "model_id": model_id,
                "manifest_revision": manifest_revision, "eval_manifest_hash": eval_manifest_hash,
            } for item, hypothesis in zip(batch, hypotheses)])], ignore_index=True)
        upload_predictions(model_id, predictions)
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()

    completed_ids = set(predictions["sample_id"].astype(str))
    if completed_ids != expected_ids:
        return {
            "model_id": model_id, "status": "partial",
            "completed": len(completed_ids), "total": len(expected_ids),
            "resume_safe": True,
        }

    ordered = frozen_eval[["sample_id", "transcript_raw"]].merge(
        predictions[["sample_id", "hypothesis"]], on="sample_id", how="left", validate="one_to_one"
    )
    metrics = asr_metrics(
        ordered["transcript_raw"].astype(str).tolist(),
        ordered["hypothesis"].astype(str).tolist(),
    ) | {
        "model_id": model_id,
        "status": "complete",
        "phase1_revision": manifest_revision,
        "eval_manifest_hash": eval_manifest_hash,
        "filter_min_duration_s": cfg.min_duration_s,
        "filter_max_words_per_s": cfg.max_words_per_s,
        "protocol_id": cfg.protocol_id,
        "pilot_limit": cfg.max_samples,
    }
    return metrics


print("GPU runner loaded. It preserves Whisper suppression defaults and forces no foreign language token.")

## 6. Run and publish metrics

In [ ]:
baseline_metrics = []
if cfg.run_baselines:
    frozen_remote = WORK_DIR / "frozen_eval_manifest.parquet"
    api.upload_file(
        repo_id=cfg.manifest_repo_id,
        repo_type="dataset",
        path_or_fileobj=str(frozen_remote),
        path_in_repo=(
            f"phase1/baselines/{cfg.protocol_id}/full/frozen_eval_manifest.parquet"
            if cfg.max_samples is None
            else f"phase1/baselines/{cfg.protocol_id}/pilot-{cfg.max_samples}/frozen_eval_manifest.parquet"
        ),
        commit_message="Freeze Dagbani WAXAL baseline evaluation manifest",
    )
    for model_id in cfg.model_ids:
        result = evaluate_model(model_id)
        baseline_metrics.append(result)
        print(json.dumps(result, indent=2))
        if result.get("status") != "complete":
            print("Session budget reached. Rerun this notebook to resume safely.")
            break

    mode = "full" if cfg.max_samples is None else f"pilot-{cfg.max_samples}"
    metrics_path = WORK_DIR / "baseline_metrics.json"
    metrics_path.write_text(json.dumps(baseline_metrics, indent=2), encoding="utf-8")
    api.upload_file(
        repo_id=cfg.manifest_repo_id,
        repo_type="dataset",
        path_or_fileobj=str(metrics_path),
        path_in_repo=f"phase1/baselines/{cfg.protocol_id}/{mode}/baseline_metrics.json",
        commit_message=f"Publish Dagbani ASR {mode} baseline metrics",
    )
    print(
        "Published private baseline evidence to "
        f"{cfg.manifest_repo_id}/phase1/baselines/{cfg.protocol_id}/{mode}"
    )
else:
    print("Baseline execution is OFF. Set cfg.run_baselines=True after enabling one Kaggle GPU.")